# 1 Initialize the Database

All the code related to data management is in the `EnvironmentData` class. This makes life easier - for example: we can send the CatsUserID once and it becomes a class property. Then, when we call other operations we don't have to send this information again.

When you create a new instance of `EnvironmentData` and there is no database, it will pull historical data and initialize the database. 

In [24]:
# Clear prior data. 
import os, sys, shutil

# Add parent directory to Python path to import EnvironmentData.
sys.path.append(os.path.dirname(os.getcwd()))

# Get the EnvironmentData class.
from EnvironmentData import EnvironmentData 

# The project adds to existing data so we need to clear that data to get a solid test from scratch.
if os.path.exists('../data'):
    shutil.rmtree('../data')
    os.makedirs('../data')

# Initialize EnvironmentData. This will run the historical data pull.
envdt = EnvironmentData(
    #days_back = 365 * 2,
    days_back = 7,
    coris_enabled = True,
    licor_enabled = True,
    conserv_enabled = True, 
    #testing = True,
    testing = False,
    # Since we are running from the experiments/ folder, we need to tell the class to use the parent directory as home.
    home_directory = ".."
)

DEBUG: Enabled data sources: ['Conserv', 'Coris', 'LI-COR']


Gathering Conserv historical data:   0%|          | 0/5 [00:00<?, ?it/s]

Gathering LI-COR readings: 100%|██████████| 33/33 [00:09<00:00,  3.39it/s]


Detailed information is saved in the log:

In [25]:
# Detailed info is saved in the log.
with open('../data/EnvironmentData.log', 'r') as file:
    for line in file.read().splitlines()[:10]:
        print(line)

2025-12-03 08:42:44,615 - EnvironmentData - INFO - Initialized Conserv client with 5 customers
2025-12-03 08:42:44,616 - EnvironmentData - INFO - Enabled data sources: ['Conserv', 'Coris', 'LI-COR']
2025-12-03 08:42:44,617 - EnvironmentData - INFO - Fetching Conserv historical data for all customers
2025-12-03 08:42:44,617 - EnvironmentData - INFO - Fetching Conserv data for period: 1764171764 to 1764776564
2025-12-03 08:42:44,620 - EnvironmentData - INFO - Fetching data for customer 333
2025-12-03 08:42:44,620 - EnvironmentData - INFO - Exporting chunk for customer 333: 2025-11-26 15:42:44+00:00 to 2025-12-03 15:42:44+00:00
2025-12-03 08:42:44,625 - EnvironmentData - INFO - Starting export for customer 333: 2025-11-26 15:42:44+00:00 to 2025-12-03 15:42:44+00:00
2025-12-03 08:42:44,626 - EnvironmentData - INFO - Conserv API POST https://api.conserv.io/v1/sensors/export headers={'x-api-key': 'XXXX', 'Content-Type': 'application/json'}
2025-12-03 08:42:45,154 - EnvironmentData - INFO - C

This saves our intermediate data to `data/sensor_readings.parquet`. 

Initially, we leave the data mostly as-is. We'll clean, add formatted dates, consolidate readings from the same device, etc. when moving to analytical steps, this preserves the source data so we can always change our mind later about how we decide to view it. 

However, at this point we are taking care to standardize the data format between different API sources. 

There are just a few columns because this is only historical data. We'll bring in current data shortly, and that will add more columns. 

In [26]:
import polars
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Coris").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764171764,1764776981,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.410004,null,true
1764172664,1764776981,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.400002,null,true
1764173564,1764776981,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.519997,null,true
1764174464,1764776981,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.43,null,true
1764175364,1764776981,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.489998,null,true


In [27]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "LI-COR").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764172800,1764777296,"""LI-COR""","""licor:10740550""","""ICSC__010C149_______""","""licor:10740550-10740550-2""","""ICSC__010C149________RH""","""RH""",null,60.068363,true
1764174600,1764777296,"""LI-COR""","""licor:10740550""","""ICSC__010C149_______""","""licor:10740550-10740550-2""","""ICSC__010C149________RH""","""RH""",null,60.379662,true
1764176400,1764777296,"""LI-COR""","""licor:10740550""","""ICSC__010C149_______""","""licor:10740550-10740550-2""","""ICSC__010C149________RH""","""RH""",null,59.055119,true
1764178200,1764777296,"""LI-COR""","""licor:10740550""","""ICSC__010C149_______""","""licor:10740550-10740550-2""","""ICSC__010C149________RH""","""RH""",null,59.525116,true
1764180000,1764777296,"""LI-COR""","""licor:10740550""","""ICSC__010C149_______""","""licor:10740550-10740550-2""","""ICSC__010C149________RH""","""RH""",null,60.135506,true


In [28]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Conserv").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764172037,1764776565,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.044006,null,true
1764172937,1764776565,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.619995,null,true
1764173837,1764776565,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.854004,null,true
1764174737,1764776565,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.800003,null,true
1764175637,1764776565,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.673996,null,true


# 2 Get Current Readings

Now we can start gathering and appending readings. There is a function `get_current_readings` that is run throughout the day, every 10 minutes for example. This function creates a parquet file at `data/new-readings` with the UTC as a filename. At the end of the day, all these readings will be consolidated into the database. 

Here is a sample of the readings:

In [29]:
# Wait 15 minutes to allow a new Conserv reading.
# import time
# time.sleep(15 * 60)  # Wait 15 minutes (900 seconds)

# envdt.get_current_readings()

# # Data is read into new-readings folder for consolidation at the end of the day.
# import os
# filename = os.listdir('../data/new-readings')[0]
# print(filename)
# polars.read_parquet('../data/new-readings/' + filename).sample(5)

# 3 Consolidate Readings

At the end of the day, new readings will be consolidated into the table. At the same time, the analytical tables will be generated. 

Analytical tables include:

* `device_readings.parquet`: Sensor readings reorganized to one row per Device and UTC, with measurements across columns vs measurements across rows.* 
* `sensors.parquet`: Information about the unique sensors. Includes information extracted from SensorName. Join this to Sensors during analysis to enhance with Building, Room, Direction, etc.
* `devices.parquet`: Information about unique devices. Includes information extracted from SensorName. 
* `utcs.parquet`: Information related to the UTC times in various datasets. Join to Sensors or Devices to enhance with Date, Time, Year, Hour, Weekday, etc.
* `sensor_readings_daily.parquet`: Example of sensor readings summarized to the daily level which reduces row count by 99.3% for even faster queries.
* `device_readings_daily.parquet`: Example of device readings summarized to the daily level which reduces row count by 99.3% for even faster queries. 

We fully re-generate analytical tables during each consolidation. The data is small enough that this is a fairly quick process, so re-running it in full each time will make it easy to ensure consistency as we expand and change the project. 

In [30]:
# To consolidate these into the database, run consolidate_readings.
envdt.consolidate_readings()

# New-readings files are gone now.
# They get deleted each day to confirm that they have been loaded into the database and prepare for the next consolidation.
if os.path.exists('../data/new-readings'):
    print(os.listdir('../data/new-readings'))

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\super\AppData\Local\Programs\Python\Python310\lib\logging\__init__.py", line 1101, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\super\AppData\Local\Programs\Python\Python310\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode characters in position 163-262: character maps to <undefined>
Call stack:
  File "C:\Users\super\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\super\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\super\Documents\arbaiza-consulting\environmental-sensor-poc\.venv\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\super\Documents

**^^ We want this to be empty** since we have consolidated new readings into the historical data. 

Once we are done working with data intake/processing, we close the class to release the file lock on the log file.

In [31]:
# When done, close the connection to the logs. 
envdt.close()

Let's look at the data we have now:

In [32]:
# Sensor Readings
# The first rows will be missing the extra fields like HexGatewayMac, etc.
#   I am pulling in some extra fields like DeviceID and DeviceName so we have that by historical. 
#   But some don't make sense to  backfill so they'll be null.
sensor_readings = polars.read_parquet('../data/sensor_readings.parquet')
sensor_readings.head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,SensorReadingUTC_SecondsFromPrior,Historical
i64,i32,str,str,str,str,str,str,f32,f32,i64,bool
1764171913,1764776667,"""Conserv""","""conserv:307:c000168""","""AOYAG_0400430_______""","""conserv:307:c000168:RH""","""AOYAG_0400430_______ - RH""","""RH""",null,52.970001,null,true
1764172828,1764776667,"""Conserv""","""conserv:307:c000168""","""AOYAG_0400430_______""","""conserv:307:c000168:RH""","""AOYAG_0400430_______ - RH""","""RH""",null,53.400002,null,true
1764173743,1764776667,"""Conserv""","""conserv:307:c000168""","""AOYAG_0400430_______""","""conserv:307:c000168:RH""","""AOYAG_0400430_______ - RH""","""RH""",null,53.279999,null,true
1764174658,1764776667,"""Conserv""","""conserv:307:c000168""","""AOYAG_0400430_______""","""conserv:307:c000168:RH""","""AOYAG_0400430_______ - RH""","""RH""",null,53.330002,null,true
1764175572,1764776667,"""Conserv""","""conserv:307:c000168""","""AOYAG_0400430_______""","""conserv:307:c000168:RH""","""AOYAG_0400430_______ - RH""","""RH""",null,53.369999,null,true


In [33]:
# Recent rows will have the full data, aside from nulls due to a sensor not providing a reading type.
sensor_readings.tail()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,SensorReadingUTC_SecondsFromPrior,Historical
i64,i32,str,str,str,str,str,str,f32,f32,i64,bool
1764767700,1764777298,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.914246,null,null,true
1764768600,1764777298,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.914246,null,null,true
1764769500,1764777298,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",70.95285,null,null,true
1764770400,1764777298,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",71.107292,null,null,true
1764771300,1764777298,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",71.261734,null,null,true


In [34]:
# Device Readings.
device_readings = polars.read_parquet('../data/device_readings.parquet')
device_readings.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,Historical,SensorReadingF,SensorReadingRh
str,str,str,str,str,str,i64,i32,bool,f32,f32
"""Conserv""","""conserv:307:c000168""","""AOYAG_0400430_______""","""conserv:307:c000168:RH, conser…","""AOYAG_0400430_______ - RH, AOY…","""RH, Temperature""",1764171913,1764776667,true,71.492004,52.970001
"""Conserv""","""conserv:307:c000168""","""AOYAG_0400430_______""","""conserv:307:c000168:RH, conser…","""AOYAG_0400430_______ - RH, AOY…","""RH, Temperature""",1764172828,1764776667,true,71.438004,53.400002
"""Conserv""","""conserv:307:c000168""","""AOYAG_0400430_______""","""conserv:307:c000168:RH, conser…","""AOYAG_0400430_______ - RH, AOY…","""RH, Temperature""",1764173743,1764776667,true,71.563995,53.279999
"""Conserv""","""conserv:307:c000168""","""AOYAG_0400430_______""","""conserv:307:c000168:RH, conser…","""AOYAG_0400430_______ - RH, AOY…","""RH, Temperature""",1764174658,1764776667,true,71.582001,53.330002
"""Conserv""","""conserv:307:c000168""","""AOYAG_0400430_______""","""conserv:307:c000168:RH, conser…","""AOYAG_0400430_______ - RH, AOY…","""RH, Temperature""",1764175572,1764776667,true,71.636002,53.369999


In [35]:
# Sensors
sensors = polars.read_parquet('../data/sensors.parquet')
sensors.head()

Source,SensorID,SensorName,SensorType,DeviceID,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection,DeviceName,SensorReadingUTC,QueryUTC,Historical
str,str,str,str,str,str,str,str,str,str,str,i64,i32,bool
"""LI-COR""","""licor:22194277-22194277-3""","""ICSC__010C140W_____F_RH""","""RH""","""licor:22194277""","""RH""","""010C140W""","""Unknown""","""F""","""Not Indicated""","""ICSC__010C140W_____F""",1764289200,1764777299,true
"""LI-COR""","""licor:22194277-22194277-3""","""ICSC__010C140W_____F_RH""","""RH""","""licor:22194277""","""RH""","""010C140W""","""Unknown""","""F""","""Not Indicated""","""ICSC__010C140W_____F""",1764316800,1764777299,true
"""LI-COR""","""licor:22194277-22194277-3""","""ICSC__010C140W_____F_RH""","""RH""","""licor:22194277""","""RH""","""010C140W""","""Unknown""","""F""","""Not Indicated""","""ICSC__010C140W_____F""",1764454200,1764777299,true
"""LI-COR""","""licor:22194277-22194277-3""","""ICSC__010C140W_____F_RH""","""RH""","""licor:22194277""","""RH""","""010C140W""","""Unknown""","""F""","""Not Indicated""","""ICSC__010C140W_____F""",1764408000,1764777299,true
"""LI-COR""","""licor:22194277-22194277-3""","""ICSC__010C140W_____F_RH""","""RH""","""licor:22194277""","""RH""","""010C140W""","""Unknown""","""F""","""Not Indicated""","""ICSC__010C140W_____F""",1764261600,1764777299,true


In [36]:
# Devices. 
devices = polars.read_parquet('../data/devices.parquet')
devices.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection
str,str,str,str,str,str,str,str,str,str,str
"""LI-COR""","""licor:22194277""","""ICSC__010C140W_____F""","""licor:22194277-22194277-3, lic…","""ICSC__010C140W_____F_RH, ICSC_…","""RH, RH, RH, RH, RH, RH, RH, RH…","""Temperature""","""010C140W""","""Unknown""","""F""","""Not Indicated"""
"""Coris""","""coris:14545""","""Yale Peabody A630 3.25.25""","""coris:25973, coris:25973, cori…","""RH 8404e5 data only, RH 8404e5…","""Humidity, Humidity, Humidity, …","""only""","""8404e5""","""Unknown""","""data""","""Not Indicated"""
"""Coris""","""coris:12394""","""Test 1 D032""","""coris:23332, coris:23332, cori…","""RH CSC H101_D032, RH CSC H101_…","""Humidity, Humidity, Humidity, …","""D032""","""CSC""","""Collection Studies Center (Wes…","""H101""","""Not Indicated"""
"""Coris""","""coris:12395""","""Test 2 0D0EA""","""coris:23330, coris:23330, cori…","""RH CSC K127_0D0EA, RH CSC K127…","""Humidity, Humidity, Humidity, …","""0D0EA""","""CSC""","""Collection Studies Center (Wes…","""K127""","""Not Indicated"""
"""Coris""","""coris:12416""","""L_0025CA0A0000CD90""","""coris:23401, coris:23401, cori…","""RH CSC K160_00cd90, RH CSC K16…","""Humidity, Humidity, Humidity, …","""00cd90""","""CSC""","""Collection Studies Center (Wes…","""K160""","""Not Indicated"""


In [37]:
# UTC Date/Time Info
utcs = polars.read_parquet('../data/utcs.parquet').head()
utcs.head()

UTC,datetime_utc,datetime_est,date,time,year,month,day_of_week,day_of_week_monday1_sunday7,hour_24,hour_12,am_pm
i64,datetime[μs],"datetime[μs, America/New_York]",date,time,i32,i8,str,i8,i8,i8,str
1764360192,2025-11-28 13:03:12,2025-11-28 08:03:12 EST,2025-11-28,08:03:12,2025,11,"""Friday""",5,8,8,"""AM"""
1764491264,2025-11-30 01:27:44,2025-11-29 20:27:44 EST,2025-11-29,20:27:44,2025,11,"""Saturday""",6,20,8,"""PM"""
1764753408,2025-12-03 02:16:48,2025-12-02 21:16:48 EST,2025-12-02,21:16:48,2025,12,"""Tuesday""",2,21,9,"""PM"""
1764753411,2025-12-03 02:16:51,2025-12-02 21:16:51 EST,2025-12-02,21:16:51,2025,12,"""Tuesday""",2,21,9,"""PM"""
1764229124,2025-11-27 00:38:44,2025-11-26 19:38:44 EST,2025-11-26,19:38:44,2025,11,"""Wednesday""",3,19,7,"""PM"""


In [38]:
# Daily Sensor Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
sensor_readings_daily = polars.read_parquet('../data/sensor_readings_daily.parquet')
sensor_readings_daily.head()

Source,date,SensorID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Conserv""",2025-12-03,"""conserv:307:c000388:RH""",1,0.0,40.849998,null,40.849998,null,40.849998
"""Conserv""",2025-12-03,"""conserv:307:c000388:Temperatur…",1,74.444,0.0,74.444,null,74.444,null
"""Conserv""",2025-12-03,"""conserv:307:c001714:RH""",1,0.0,49.650002,null,49.650002,null,49.650002
"""Conserv""",2025-12-03,"""conserv:307:c001714:Temperatur…",1,71.977997,0.0,71.977997,null,71.977997,null
"""Conserv""",2025-12-03,"""conserv:307:c002441:RH""",1,0.0,54.439999,null,54.439999,null,54.439999


In [39]:
# Daily Device Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
device_readings_daily = polars.read_parquet('../data/device_readings_daily.parquet')
device_readings_daily.head()

Source,date,DeviceID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Conserv""",2025-12-03,"""conserv:307:c000388""",1,74.444,40.849998,74.444,40.849998,74.444,40.849998
"""Conserv""",2025-12-03,"""conserv:307:c001714""",1,71.977997,49.650002,71.977997,49.650002,71.977997,49.650002
"""Conserv""",2025-12-03,"""conserv:307:c002441""",1,70.574005,54.439999,70.574005,54.439999,70.574005,54.439999
"""Conserv""",2025-12-03,"""conserv:307:c007108""",1,64.958,55.830002,64.958,55.830002,64.958,55.830002
"""Conserv""",2025-12-03,"""conserv:307:c007112""",1,67.460007,50.939999,67.460007,50.939999,67.460007,50.939999


In [40]:
# Differentiate historical vs. cron readings by filtering on Historical = true.
import duckdb
duckdb.sql("""
    SELECT *
    FROM read_parquet('../data/device_readings.parquet') 
    WHERE Historical
    LIMIT 5
""").to_df()

,Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,Historical,SensorReadingF,SensorReadingRh
0,Conserv,conserv:307:c000168,AOYAG_0400430_______,"conserv:307:c000168:RH, conserv:307:c000168:Te...","AOYAG_0400430_______ - RH, AOYAG_0400430______...","RH, Temperature",1764171913,1764776667,True,71.492004,52.970001
1,Conserv,conserv:307:c000168,AOYAG_0400430_______,"conserv:307:c000168:RH, conserv:307:c000168:Te...","AOYAG_0400430_______ - RH, AOYAG_0400430______...","RH, Temperature",1764172828,1764776667,True,71.438004,53.400002
2,Conserv,conserv:307:c000168,AOYAG_0400430_______,"conserv:307:c000168:RH, conserv:307:c000168:Te...","AOYAG_0400430_______ - RH, AOYAG_0400430______...","RH, Temperature",1764173743,1764776667,True,71.563995,53.279999
3,Conserv,conserv:307:c000168,AOYAG_0400430_______,"conserv:307:c000168:RH, conserv:307:c000168:Te...","AOYAG_0400430_______ - RH, AOYAG_0400430______...","RH, Temperature",1764174658,1764776667,True,71.582001,53.330002
4,Conserv,conserv:307:c000168,AOYAG_0400430_______,"conserv:307:c000168:RH, conserv:307:c000168:Te...","AOYAG_0400430_______ - RH, AOYAG_0400430______...","RH, Temperature",1764175572,1764776667,True,71.636002,53.369999


In [41]:
duckdb.sql("""SELECT DISTINCT
    sr.Source,
    u.datetime_est
FROM '../data/sensor_readings.parquet' sr
LEFT JOIN '../data/utcs.parquet' u 
    ON sr.SensorReadingUTC = u.utc
WHERE sr.Source = 'LI-COR'
ORDER BY sr.SensorReadingUTC
""").to_df()

,Source,datetime_est
0,LI-COR,2025-11-26 01:45:00-07:00
1,LI-COR,2025-11-26 01:50:00-07:00
2,LI-COR,2025-11-26 02:00:00-07:00
3,LI-COR,2025-11-26 02:10:00-07:00
4,LI-COR,2025-11-26 02:15:00-07:00
...,...,...
1256,LI-COR,2025-12-03 00:00:00-07:00
1257,LI-COR,2025-12-03 00:15:00-07:00
1258,LI-COR,2025-12-03 00:30:00-07:00
1259,LI-COR,2025-12-03 01:00:00-07:00


# Validation & Alerts

There are two diagnostic files we can review to see if there are alerts or errors. 

In [42]:
# Read ../data/validation-results.csv
import pandas as pd
validation_results = pd.read_csv('../data/validation-results.csv')
validation_results

,run_datetime_est,run_utc,test_name,result,details
0,2025-12-03 10:55:05 EST,1764777305,required_columns_present,PASS,All 11 required columns are present in sensor ...
1,2025-12-03 10:55:05 EST,1764777305,column_data_types,PASS,All columns have the expected data types (e.g....
2,2025-12-03 10:55:05 EST,1764777305,non_null_values,PASS,Every sensor reading row has at least one non-...
3,2025-12-03 10:55:05 EST,1764777305,no_duplicate_readings,PASS,No duplicate readings found. Each sensor has u...
4,2025-12-03 10:55:05 EST,1764777305,sensor_name_consistency,PASS,All sensors have consistent names across all t...
5,2025-12-03 10:55:05 EST,1764777305,reading_interval_check,PASS,All consecutive readings are within 15 minutes...
6,2025-12-03 10:55:05 EST,1764777305,data_gaps_LI-COR,WARN,Found 2680 gaps in LI-COR data where readings ...
7,2025-12-03 10:55:05 EST,1764777305,data_gaps_Conserv,WARN,Found 3286 gaps in Conserv data where readings...
8,2025-12-03 10:55:05 EST,1764777305,alerts_Coris,PASS,No alerts triggered for Coris. All sensor read...
9,2025-12-03 10:55:05 EST,1764777305,alerts_LI-COR,PASS,No alerts triggered for LI-COR. All sensor rea...


In [43]:
# Validation results that did not pass.
validation_results[validation_results['result'] != "PASS"]

,run_datetime_est,run_utc,test_name,result,details
6,2025-12-03 10:55:05 EST,1764777305,data_gaps_LI-COR,WARN,Found 2680 gaps in LI-COR data where readings ...
7,2025-12-03 10:55:05 EST,1764777305,data_gaps_Conserv,WARN,Found 3286 gaps in Conserv data where readings...


In [44]:
# Read ../data/alerts.csv
alerts = pd.read_csv('../data/alerts.csv')
alerts.sample(15).sort_values(by='event_utc')

,event,Source,SensorID,SensorName,event_utc,event_datetime_est,event_end_utc,event_end_datetime_est,gap_minutes,reading_type,reading_value,threshold_min,threshold_max,detected_utc,detected_datetime_est
4966,DATA_GAP,LI-COR,licor:22040302-22040302-2,ICSC__010C145S_______RH,1764181800,2025-11-26 13:30:00 EST,1764183600,2025-11-26 14:00:00 EST,30.0,NaN,NaN,NaN,NaN,1764777305,2025-12-03 10:55:05 EST
2148,DATA_GAP,Conserv,conserv:307:c008940:RH,ACSC__010J126N___s1__ - RH,1764193084,2025-11-26 16:38:04 EST,1764195784,2025-11-26 17:23:04 EST,45.0,NaN,NaN,NaN,NaN,1764777305,2025-12-03 10:55:05 EST
4977,DATA_GAP,LI-COR,licor:22040302-22040302-2,ICSC__010C145S_______RH,1764201600,2025-11-26 19:00:00 EST,1764203400,2025-11-26 19:30:00 EST,30.0,NaN,NaN,NaN,NaN,1764777305,2025-12-03 10:55:05 EST
3648,DATA_GAP,LI-COR,licor:10740550-10740550-2,ICSC__010C149________RH,1764221400,2025-11-27 00:30:00 EST,1764223200,2025-11-27 01:00:00 EST,30.0,NaN,NaN,NaN,NaN,1764777305,2025-12-03 10:55:05 EST
3352,DATA_GAP,LI-COR,licor:10740550-10740550-1,ICSC__010C149________Temperature,1764291600,2025-11-27 20:00:00 EST,1764293400,2025-11-27 20:30:00 EST,30.0,NaN,NaN,NaN,NaN,1764777305,2025-12-03 10:55:05 EST
3174,DATA_GAP,Conserv,conserv:333:c008785:Temperature,BYCBA_040040104_S___ - Temperature,1764459065,2025-11-29 18:31:05 EST,1764462665,2025-11-29 19:31:05 EST,60.0,NaN,NaN,NaN,NaN,1764777305,2025-12-03 10:55:05 EST
4459,DATA_GAP,LI-COR,licor:10889153-10889153-2,C145 Structural_RH,1764475200,2025-11-29 23:00:00 EST,1764477000,2025-11-29 23:30:00 EST,30.0,NaN,NaN,NaN,NaN,1764777305,2025-12-03 10:55:05 EST
3242,DATA_GAP,Conserv,conserv:333:c009063:Temperature,BYCBA_B100B06_______ - Temperature,1764576378,2025-12-01 03:06:18 EST,1764578178,2025-12-01 03:36:18 EST,30.0,NaN,NaN,NaN,NaN,1764777305,2025-12-03 10:55:05 EST
525,DATA_GAP,Conserv,conserv:307:c007106:Temperature,c007106 - OYAG2 - 2019.57.1 - Temperature,1764585373,2025-12-01 05:36:13 EST,1764587173,2025-12-01 06:06:13 EST,30.0,NaN,NaN,NaN,NaN,1764777305,2025-12-03 10:55:05 EST
3853,DATA_GAP,LI-COR,licor:10740550-10740550-2,ICSC__010C149________RH,1764590400,2025-12-01 07:00:00 EST,1764592200,2025-12-01 07:30:00 EST,30.0,NaN,NaN,NaN,NaN,1764777305,2025-12-03 10:55:05 EST


In [45]:
# Group alerts by Source ans event.
alerts.groupby(['Source', 'event']).size().reset_index(name='count')

# No alerts (only data gaps) means all readings were within thresholds.

,Source,event,count
0,Conserv,DATA_GAP,3286
1,LI-COR,DATA_GAP,2680


Now you are ready to move onto analysis to get human-readable results (not indexed by UTC timestamps). See 2-examples-analysis.ipynb.